<h1 style="color:purple; text-align:center;">
Desafios del satelite de Galileo
</h1>

In [1]:
# Importamos sqlite para trabajar y enlazamos con la tabla de datos.

import sqlite3
import pandas as pd

con = sqlite3.connect("biblioteca.db")

### Desafío 1: Sistema de multas

In [ ]:
# 1. Añade una tabla de multas para préstamos con retraso

pd.read_sql("""CREATE TABLE multas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    prestamo_id INTEGER NOT NULL,
    importe REAL NOT NULL CHECK (importe >= 0),
    pagada BOOLEAN DEFAULT 0,
    FOREIGN KEY (prestamo_id) REFERENCES prestamos(id)
);""", con)

In [3]:
# Inserta multas calculando 0.50€ por día de retraso.

multas = pd.read_sql("""SELECT
	p.id,
    (JULIANDAY(p.fecha_devolucion_real) - JULIANDAY(p.fecha_devolucion_prevista)) * 0.50 AS dias_retraso
FROM prestamos p
WHERE p.fecha_devolucion_real > p.fecha_devolucion_prevista;""", con)

multas

,id,dias_retraso
0,4,378.0


### Desafío 2: Historial completo de usuario

Crea una consulta que muestre:

- Todos los préstamos de un usuario (activos + devueltos)
- Total de días que ha tenido libros
- Número de retrasos
- Total de multas (si implementaste el desafío 1)

In [4]:
prestamos_usuario = pd.read_sql("""SELECT 
	usuario_ID, 
	COUNT(libro_ID) AS num_libros, 
	sum(dias_prestamo) AS total_dias,
	SUM((p.devuelto = 1) AND (p.fecha_devolucion_real > p.fecha_devolucion_prevista)) AS num_retrasos,
	(SELECT importe FROM multas WHERE multas.prestamo_id = p.id) AS importe_multas
FROM Prestamos as p
WHERE usuario_ID = 1
GROUP BY p.usuario_id""", con)

prestamos_usuario

,usuario_id,num_libros,total_dias,num_retrasos,importe_multas
0,1,3,57,0,None


## Desafío 3: Recomendaciones

Encuentra libros del mismo autor que los que ha leído un usuario pero que aún no ha tomado prestados.

In [5]:
recomendaciones = pd.read_sql("""SELECT lb.ID, 
	lb.titulo, 
	at.nombre as ESCRITOR -- columnas que queremos
FROM Libros as lb -- de que tabla principal
INNER JOIN Autores as at -- añadimos los autores a la principal
	ON lb.autor_id = at.ID -- nexo de union
WHERE lb.autor_id IN -- Consulta 2: autores de los libros que el usuario ya leyó
	(SELECT lb.autor_id
	FROM Prestamos as pt
	INNER JOIN libros as lb
		ON pt.libro_id = lb.id
		WHERE pt.usuario_id = 1)
AND lb.id NOT IN ( --solo los libros del mismo autor que NO ha leído
        SELECT libro_id
        FROM Prestamos
        WHERE usuario_id = 1)""", con)

recomendaciones

,ID,titulo,ESCRITOR
0,6,El amor en los tiempos del cólera,Gabriel García Márquez
